# Native RAG Projesi (framework'suz, sıfırdan)

Görev şuydu: LangChain/LlamaIndex gibi hazır RAG kütüphaneleri kullanmadan, kendi
elimle bir RAG (Retrieval-Augmented Generation) hattı kurmak. Yani dört şeyi kendim
yazmam gerekiyordu: parçalama, vektörleştirme, benzerlik araması ve cevap üretme.

Bilgi kaynağı olarak kendi yazdığım **Bölüm 4 - RAG Mimarileri** araştırma notumu
(PDF) kullandım — hem gerçek bir metin oluyor hem de içeriğini bildiğim için "belge
içi" ve "belge dışı" soruları test etmek daha kolay oldu.

Embedding ve cevap üretme (generation) için **Gemini API** kullandım (key'i ayrı bir
`.env` dosyasından okuyorum, koda hiç yazmadım).

## 1) Metni yükleme ve parçalama (chunking)

Önce PDF'ten ham metni çıkardım, sonra kendi yazdığım `parcala()` fonksiyonuyla
sabit boyutlu (600 karakter) ve aralarında 100 karakterlik örtüşme (overlap) olan
parçalara böldüm. Örtüşme bıraktım çünkü bölme noktasında bir cümlenin ortadan
kesilip anlamının kaybolmasını istemedim.

In [1]:
import fitz  # PyMuPDF, PDF'ten metin cikarmak icin
import urllib.request
import json
import math

with open(".env", encoding="utf-8") as f:
    API_KEY = f.read().strip().split("=", 1)[1]

EMBED_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:embedContent?key={}"
GEN_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent?key={}"


def yukle_metin(pdf_yolu):
    doc = fitz.open(pdf_yolu)
    metin = ""
    for sayfa in doc:
        metin += sayfa.get_text()
    return metin


def parcala(metin, parca_boyutu=600, ortusme=100):
    parcalar = []
    basla = 0
    while basla < len(metin):
        bitis = basla + parca_boyutu
        parcalar.append(metin[basla:bitis].strip())
        basla = bitis - ortusme
    return [p for p in parcalar if p]


metin = yukle_metin("bolum_4_rag_mimarileri.pdf")
parcalar = parcala(metin, parca_boyutu=600, ortusme=100)
print(f"toplam karakter: {len(metin)}, parca sayisi: {len(parcalar)}")
for i, p in enumerate(parcalar):
    print(f"  parca {i}: {len(p)} karakter")

toplam karakter: 4530, parca sayisi: 10
  parca 0: 600 karakter
  parca 1: 600 karakter
  parca 2: 600 karakter
  parca 3: 600 karakter
  parca 4: 600 karakter
  parca 5: 600 karakter
  parca 6: 599 karakter
  parca 7: 600 karakter
  parca 8: 529 karakter
  parca 9: 29 karakter


## 2) Vektörleştirme (embedding)

Her parçayı Gemini'nin `gemini-embedding-001` modeline gönderip 3072 boyutlu gerçek
bir sayı vektörü aldım. Burada "API kullanma" şartını Gemini'nin embedding
endpoint'ine ham HTTP isteği atarak karşıladım — hazır bir RAG kütüphanesi değil,
sadece bir embedding servisi çağırdım.

In [2]:
def embed_al(metin):
    body = json.dumps({"content": {"parts": [{"text": metin}]}}).encode("utf-8")
    req = urllib.request.Request(
        EMBED_URL.format(API_KEY), data=body, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    return data["embedding"]["values"]


parca_vektorleri = []
for i, p in enumerate(parcalar):
    v = embed_al(p)
    parca_vektorleri.append(v)
    print(f"  parca {i} embed edildi, boyut={len(v)}")

  parca 0 embed edildi, boyut=3072
  parca 1 embed edildi, boyut=3072
  parca 2 embed edildi, boyut=3072
  parca 3 embed edildi, boyut=3072
  parca 4 embed edildi, boyut=3072
  parca 5 embed edildi, boyut=3072
  parca 6 embed edildi, boyut=3072
  parca 7 embed edildi, boyut=3072
  parca 8 embed edildi, boyut=3072
  parca 9 embed edildi, boyut=3072


## 3) Saf benzerlik araması (native similarity search)

Burası işin "sıfırdan" kısmı — hiçbir vektör veritabanı ya da hazır kütüphane
fonksiyonu kullanmadım, kosinüs benzerliğini formülünden kendim yazdım:
`(a·b) / (|a|·|b|)`. Bir soru geldiğinde onu da embed'leyip, elimdeki 10 parça
vektörüyle tek tek karşılaştırıp en yüksek skorlu 3 parçayı seçiyorum.

In [3]:
def kosinus_benzerlik(a, b):
    ic_carpim = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return ic_carpim / (norm_a * norm_b)


def en_yakin_parcalari_bul(soru, parcalar, parca_vektorleri, k=3):
    soru_vektor = embed_al(soru)
    skorlar = [
        (kosinus_benzerlik(soru_vektor, pv), i) for i, pv in enumerate(parca_vektorleri)
    ]
    skorlar.sort(reverse=True)
    return [(parcalar[i], skor) for skor, i in skorlar[:k]]

## 4) İstem ve üretim (prompting & generation)

Bulunan parçaları soruyla birleştirip Gemini'nin `gemini-flash-latest` modeline
gönderiyorum. Prompt'un içine bilerek şu kuralı koydum: **"sadece bu parçalardaki
bilgiyi kullan, yoksa 'Bilgi bulunamadı.' yaz"** — halüsinasyonu engellemenin en
basit yolu bu, araştırma bölümünde de okumuştum.

In [4]:
def cevap_uret(soru, baglam_parcalari):
    baglam = "\n\n---\n\n".join(baglam_parcalari)
    prompt = f"""Aşağıda bir metinden alınmış parçalar var. SADECE bu parçalardaki bilgiyi kullanarak soruyu cevapla. Eğer cevap parçalarda yoksa, başka hiçbir şey söylemeden sadece "Bilgi bulunamadı." yaz.

BAĞLAM:
{baglam}

SORU: {soru}

CEVAP:"""
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode("utf-8")
    req = urllib.request.Request(
        GEN_URL.format(API_KEY), data=body, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    return data["candidates"][0]["content"]["parts"][0]["text"]

## 5) Test 1 — belge içi soru

Bölüm 4'te gerçekten yazdığım bir konu hakkında soru sordum: GraphRAG ne zaman
avantajlı oluyor? Cevabın metinden gelmesini bekliyorum.

In [5]:
soru1 = "GraphRAG ne zaman avantajli, hangi tip sorularda vektor tabanli aramadan daha iyi calisiyor?"
bulunanlar1 = en_yakin_parcalari_bul(soru1, parcalar, parca_vektorleri, k=3)
print("Soru:", soru1)
for p, skor in bulunanlar1:
    print(f"  [skor={skor:.4f}] {p[:80]}...")

cevap1 = cevap_uret(soru1, [p for p, s in bulunanlar1])
print("\nCEVAP 1:\n", cevap1)

Soru: GraphRAG ne zaman avantajli, hangi tip sorularda vektor tabanli aramadan daha iyi calisiyor?
  [skor=0.8356] ltülü oluyor.
Veri Yapısına Göre Değişen RAG'ler
Vector-based RAG'de her parça k...
  [skor=0.7613] mıyor ama graf ilişkileri takip edebildiği
için bu tarz sorularda çok daha güçlü...
  [skor=0.7533] kullanıcı sorusu kötü ifade edilirse arama
tutmuyor, getirilen parçalar alakasız...

CEVAP 1:
 GraphRAG, "X şirketinin CEO'sunun daha önce çalıştığı şirket hangi sektördeydi?" gibi ilişkilerin takip edilmesi gereken çok adımlı sorularda avantajlıdır. Vektör tabanlı arama bu tür zincirleme bilgileri tek bir parçada bulamazken, GraphRAG graf ilişkilerini takip edebildiği için çok adımlı sorularda daha güçlü çalışmaktadır.


Skorlara bakınca da mantıklı — en yüksek skor (0.8356) gerçekten GraphRAG'ın
anlatıldığı parçaya çıkmış. Cevap da uydurma değil, doğrudan metindeki bilgiyi
kullanmış.

## 6) Test 2 — belge dışı soru

Şimdi de bölümde hiç geçmeyen, tamamen alakasız bir soru sordum: Python'da for
döngüsü nasıl yazılır? Bu metinde bunun cevabı yok, model "Bilgi bulunamadı."
demeli — halüsinasyon yapmamalı.

In [6]:
soru2 = "Python programlama dilinde bir for dongusu nasil yazilir?"
bulunanlar2 = en_yakin_parcalari_bul(soru2, parcalar, parca_vektorleri, k=3)
print("Soru:", soru2)
for p, skor in bulunanlar2:
    print(f"  [skor={skor:.4f}] {p[:80]}...")

cevap2 = cevap_uret(soru2, [p for p, s in bulunanlar2])
print("\nCEVAP 2:\n", cevap2)

Soru: Python programlama dilinde bir for dongusu nasil yazilir?
  [skor=0.5551] ük bir temperature kullanmak....
  [skor=0.5498] ARAŞTIRMA NOTLARIM · BÖLÜM 4
RAG Mimarileri ve Araştırması
Bu bölüm en yoğun kıs...
  [skor=0.5417] mıyor ama graf ilişkileri takip edebildiği
için bu tarz sorularda çok daha güçlü...

CEVAP 2:
 Bilgi bulunamadı.


## Değerlendirme

Burada dikkatimi çeken şey skorlardaki fark oldu: belge içi soruda en iyi skor
0.83'tü, belge dışı soruda ise en iyisi bile 0.55'te kaldı — yani soru gerçekten
alakasız olunca benzerlik skoru da düşüyor, sistem bunu "fark edebiliyor". Ama asıl
halüsinasyonu engelleyen şey skor değil, prompt'a koyduğum "sadece bağlamdan
cevap ver, yoksa bilgi bulunamadı de" kuralıydı — düşük skorlu parçaları da olsa
olsa modele gönderiyorum, halüsinasyonu asıl engelleyen istem (prompt) tarafındaki
bu kısıtlama oluyor.

Yani sonuç olarak dört adımı da (chunking, embedding, similarity search,
generation) hiçbir RAG kütüphanesi kullanmadan, kendi yazdığım fonksiyonlarla
kurmuş oldum ve gerçek sorularla test ettim.